1) 충성 사용자 리포트
이용자와 댓글 데이터를 결합해, 총 댓글 수가 10개 이상인 이용자만 추립니다.
각 이용자에 대해 다음 지표를 계산해 주세요:
commentsCount(총 댓글 수)
avgTextLen(댓글 평균 길이, null/missing은 길이 0으로 간주)
lastCommentDate(가장 최근 댓글 시점)
결과에는 **이름(name), 이메일(email)**만 식별 정보로 포함하고, 위 활동 지표 **내림차순(많이·길게·최근 순)**으로 정렬해 상위 이용자가 누구인지 한눈에 보이도록 합니다.
2) 영화 인사이트 리포트
영화 컬렉션만으로 한 번의 파이프라인에서 다음을 동시에 산출해 주세요.
최신 영화 5편: 제목(title)과 연도(year) 목록
고평점 영화 개수: imdb.rating ≥ 8 작품의 총 개수(숫자 1개)
장르별 상위 10: 장르별 작품 수를 집계해 상위 10개를 많이 나온 순으로 정렬
연도별 평균 평점(최근 10년): 최신 연도를 기준으로 최근 10개 연도의 평균 평점을 산출(연도-오름차순 표기)
출력 형식 가이드
충성 사용자 리포트: 리스트(배열) 형태. 각 원소는 {name, email, commentsCount, avgTextLen, lastCommentDate}.
영화 인사이트 리포트: 단일 문서(오브젝트) 형태.
키는 latest5, highRatedCount, genresTop10, yearlyAvgRecent10로 구성.
highRatedCount는 숫자 하나로 제출

In [7]:
from pymongo import MongoClient 

client = MongoClient("mongodb://localhost:27017")

db = client.sample_mflix
users = db.users
movies = db.movies


In [12]:
# 이용자와 댓글 데이터를 결합해, 총 댓글 수가 10개 이상인 이용자만 추립니다.
# 각 이용자에 대해 다음 지표를 계산해 주세요:
# commentsCount(총 댓글 수)
# avgTextLen(댓글 평균 길이, null/missing은 길이 0으로 간주)
# lastCommentDate(가장 최근 댓글 시점)

pipeline = [
    {"$lookup" :{
     "from" : "comments",
     "localField": "name",
     "foreignField": "name",
     "as": "UC"}},
    {"$unwind" : "$UC"},
    {"$group":{
        "_id" : "$name",
        "commentsCount": {"$sum":1},
        "avgTextLen": {"$avg":{"$strLenCP":{"$ifNull": ["$UC.text", "0"]}}},
        "lastCommentDate" : {
            "$max":"$UC.date"
        }
        }
    },
     {"$match": {"commentsCount": {"$gte": 10}}},
    {
        "$project":{
            "_id":0,
            "name":"$_id",
            "commentsCount":1,
            "avgTextLen" : 1,
            "lastCommentDate":1
        }
    }
    
]

for user in users.aggregate(pipeline):
    print(user)


{'commentsCount': 282, 'avgTextLen': 152.32624113475177, 'lastCommentDate': datetime.datetime(2017, 7, 3, 22, 41, 6), 'name': 'Keith Phillips'}
{'commentsCount': 262, 'avgTextLen': 154.19847328244273, 'lastCommentDate': datetime.datetime(2017, 6, 11, 5, 53, 28), 'name': 'Podrick Payne'}
{'commentsCount': 304, 'avgTextLen': 150.8815789473684, 'lastCommentDate': datetime.datetime(2017, 8, 9, 1, 29, 28), 'name': 'Thoros of Myr'}
{'commentsCount': 258, 'avgTextLen': 150.62403100775194, 'lastCommentDate': datetime.datetime(2017, 9, 9, 16, 58, 34), 'name': 'Michael Moore'}
{'commentsCount': 247, 'avgTextLen': 149.67206477732793, 'lastCommentDate': datetime.datetime(2016, 11, 15, 20, 19, 39), 'name': 'Christopher Robinson'}
{'commentsCount': 257, 'avgTextLen': 154.3385214007782, 'lastCommentDate': datetime.datetime(2017, 5, 27, 12, 45, 54), 'name': 'Ms. Cathy Miller'}
{'commentsCount': 259, 'avgTextLen': 150.52509652509653, 'lastCommentDate': datetime.datetime(2017, 8, 21, 6, 24, 14), 'name':

NameError: name 'name' is not defined

In [31]:
# 영화 컬렉션만으로 한 번의 파이프라인에서 다음을 동시에 산출해 주세요.
# 최신 영화 5편: 제목(title)과 연도(year) 목록
# 고평점 영화 개수: imdb.rating ≥ 8 작품의 총 개수(숫자 1개)
# 장르별 상위 10: 장르별 작품 수를 집계해 상위 10개를 많이 나온 순으로 정렬
# 연도별 평균 평점(최근 10년): 최신 연도를 기준으로 최근 10개 연도의 평균 평점을 산출(연도-오름차순 표기)
# 영화 인사이트 리포트: 단일 문서(오브젝트) 형태.
# 키는 latest5, highRatedCount, genresTop10, yearlyAvgRecent10로 구성.
# highRatedCount는 숫자 하나로 제출


pipeline = [
    {"$facet":{
        "latest5" :[
            {"$sort" :{"released" : -1}},
            {"$project": {"_id": 0,"title": 1,"year":1}},
            {"$limit":5}
        ],
        "highRatedCount" :[
            {"$match": {
                    "imdb.rating": {"$type": "number", "$gte": 8}
                }},
            {"$count": "n"}
        ],
        "genresTop10" : [
            {"$unwind" : "$genres"},
            {"$group" : {
                "_id" : "$genres",
                "Top10" : {"$sum":1}
            }},
            {"$sort" : {"Top10":-1}},
            {"$project":{"_id": 1,"Top10":1}}
        ],
         "yearlyAvgRecent10": [
             {"$group": {
                "_id":"$year",
                "avgRating": {"$avg": "$imdb.rating"},
                 "count": {"$sum": 1}
             }},
             {"$sort":{"avgRating" : -1}},{"$limit" : 10}
         ]
    }},
    {"$project":{
        "latest5":1,
        "highRatedCount":1,
        "genresTop10":1,
        "yearlyAvgRecent10":1
        
        
    }}
]

for movie in movies.aggregate(pipeline):
    print(movie)

{'latest5': [{'title': 'The Treasure', 'year': 2015}, {'year': 2015, 'title': 'Knight of Cups'}, {'title': 'Sand Castles', 'year': 2014}, {'title': 'Shut In', 'year': 2015}, {'title': 'Dègradè', 'year': 2015}], 'highRatedCount': [{'n': 1596}], 'genresTop10': [{'_id': 'Drama', 'Top10': 13789}, {'_id': 'Comedy', 'Top10': 7024}, {'_id': 'Romance', 'Top10': 3665}, {'_id': 'Crime', 'Top10': 2678}, {'_id': 'Thriller', 'Top10': 2658}, {'_id': 'Action', 'Top10': 2539}, {'_id': 'Documentary', 'Top10': 2129}, {'_id': 'Adventure', 'Top10': 2045}, {'_id': 'Horror', 'Top10': 1703}, {'_id': 'Biography', 'Top10': 1404}, {'_id': 'Family', 'Top10': 1311}, {'_id': 'Mystery', 'Top10': 1259}, {'_id': 'Fantasy', 'Top10': 1153}, {'_id': 'Sci-Fi', 'Top10': 1034}, {'_id': 'History', 'Top10': 999}, {'_id': 'Animation', 'Top10': 971}, {'_id': 'Music', 'Top10': 840}, {'_id': 'War', 'Top10': 794}, {'_id': 'Musical', 'Top10': 487}, {'_id': 'Short', 'Top10': 478}, {'_id': 'Sport', 'Top10': 390}, {'_id': 'Western', 

In [8]:
# 영화 컬렉션만으로 한 번의 파이프라인에서 다음을 동시에 산출해 주세요.
# 최신 영화 5편: 제목(title)과 연도(year) 목록
# 고평점 영화 개수: imdb.rating ≥ 8 작품의 총 개수(숫자 1개)
# 장르별 상위 10: 장르별 작품 수를 집계해 상위 10개를 많이 나온 순으로 정렬
# 연도별 평균 평점(최근 10년): 최신 연도를 기준으로 최근 10개 연도의 평균 평점을 산출(연도-오름차순 표기)
# 영화 인사이트 리포트: 단일 문서(오브젝트) 형태.
# 키는 latest5, highRatedCount, genresTop10, yearlyAvgRecent10로 구성.
# highRatedCount는 숫자 하나로 제출


pipeline = [
    {"$facet":{
        "latest5" :[
            {"$sort" :{"released" : -1}},
            {"$project": {"_id": 0,"title": 1,"year":1}},
            {"$limit":5}
        ],
        "highRatedCount" :[
            {"$match": {
                    "imdb.rating": {"$type": "number", "$gte": 8}
                }},
            {"$count": "n"}
        ],
        "genresTop10" : [
            {"$unwind" : "$genres"},
            {"$group" : {
                "_id" : "$genres",
                "Top10" : {"$sum":1}
            }},
            {"$sort" : {"Top10":-1}},
            {"$project":{"_id": 1,"Top10":1}}
        ],
         "yearlyAvgRecent10": [
             {"$group": {
                "_id":"$year",
                "avgRating": {"$avg": "$imdb.rating"},
                 "count": {"$sum": 1}
             }},
             {"$sort":{"avgRating" : -1}},{"$limit" : 10}
         ]
    }},
    {"$project":{
        "latest5":1,
        "highRatedCount":1,
        "genresTop10":1,
        "yearlyAvgRecent10":1
        
        
    }}
]
top = list(movies.aggregate(pipeline))
top
  

[{'latest5': [{'title': 'The Treasure', 'year': 2015},
   {'year': 2015, 'title': 'Knight of Cups'},
   {'title': 'Sand Castles', 'year': 2014},
   {'title': 'Shut In', 'year': 2015},
   {'title': 'Dègradè', 'year': 2015}],
  'highRatedCount': [{'n': 1596}],
  'genresTop10': [{'_id': 'Drama', 'Top10': 13789},
   {'_id': 'Comedy', 'Top10': 7024},
   {'_id': 'Romance', 'Top10': 3665},
   {'_id': 'Crime', 'Top10': 2678},
   {'_id': 'Thriller', 'Top10': 2658},
   {'_id': 'Action', 'Top10': 2539},
   {'_id': 'Documentary', 'Top10': 2129},
   {'_id': 'Adventure', 'Top10': 2045},
   {'_id': 'Horror', 'Top10': 1703},
   {'_id': 'Biography', 'Top10': 1404},
   {'_id': 'Family', 'Top10': 1311},
   {'_id': 'Mystery', 'Top10': 1259},
   {'_id': 'Fantasy', 'Top10': 1153},
   {'_id': 'Sci-Fi', 'Top10': 1034},
   {'_id': 'History', 'Top10': 999},
   {'_id': 'Animation', 'Top10': 971},
   {'_id': 'Music', 'Top10': 840},
   {'_id': 'War', 'Top10': 794},
   {'_id': 'Musical', 'Top10': 487},
   {'_id': '